# Analisis Mikrobioma 16S rRNA: NCBI sampai Analisis Diversitas

Notebook ini mengotomatisasi pipeline praktikum untuk **seluruh 26 run** pada BioProject `PRJNA1088555`. Tahapan dibuat berurutan, menyimpan manifest pada setiap checkpoint, dan dapat dilanjutkan tanpa mengulang hasil yang sudah tersedia.

> Jalankan sel dari atas ke bawah. Gunakan `MODE = 'uji'` saat mencoba notebook, lalu ubah menjadi `MODE = 'lengkap'` untuk memproses semua sampel.


## Peta Pipeline

1. Mengambil metadata dan 26 accession dari NCBI.
2. Memastikan empat kelompok sampel sesuai metadata.
3. Mengunduh FASTQ dan memvalidasi berkas.
4. Menyiapkan FASTA untuk paired-end dan single-end.
5. Menjalankan `Summary.seqs`, `Screen.seqs`, dan `Unique.seqs`.
6. Mengirim **screened reads (`good.fasta`)** ke Kraken2 di Galaxy Europe.
7. Menjalankan Bracken hanya jika database/distribusinya kompatibel dengan Kraken2.
8. Membuat Krona, matriks kelimpahan, indeks Shannon, dan Bray-Curtis.

Setiap tahap menghasilkan CSV manifest di folder `metadata`. Manifest menjadi catatan sampel yang berhasil, dilewati, atau gagal.


In [ ]:
# 1. Memuat pustaka yang diperlukan
from pathlib import Path
from IPython.display import display
from io import StringIO
import gzip
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
import urllib.request
import zipfile
import time

try:
    import pandas as pd
    import requests
    from tqdm.auto import tqdm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'requests', 'tqdm'])
    import pandas as pd
    import requests
    from tqdm.auto import tqdm

IS_COLAB = 'google.colab' in sys.modules
SYSTEM = platform.system()
ENVIRONMENT_NAME = 'Google Colab' if IS_COLAB else SYSTEM
print(f'Lingkungan : {ENVIRONMENT_NAME}')
print(f'Python     : {platform.python_version()}')
print('[OK] Tahap selesai.')


## 1. Konfigurasi

Pada percobaan pertama, pertahankan `TEST_MODE = True`. Notebook hanya mengunduh satu run. Setelah alur berhasil, jumlah run dapat ditambah secara bertahap.


In [ ]:
# 2. Pengaturan utama
BIOPROJECT = 'PRJNA1088555'
MODE = 'lengkap'  # Pilihan: 'uji' atau 'lengkap'
if MODE not in {'uji', 'lengkap'}:
    raise ValueError("MODE harus 'uji' atau 'lengkap'.")
TEST_MODE = MODE == 'uji'
MAX_RUNS = 1
THREADS = min(4, os.cpu_count() or 1)

# Pada Colab, hasil disimpan di /content. Pada komputer lokal, hasil
# disimpan di folder yang sama dengan lokasi notebook dijalankan.
ROOT = Path('/content/mikrobioma_ncbi') if IS_COLAB else Path.cwd() / 'mikrobioma_ncbi'
DIR_METADATA = ROOT / 'metadata'
DIR_SRA = ROOT / 'sra_cache'
DIR_FASTQ = ROOT / 'fastq'
DIR_TEMP = ROOT / 'temp'
DIR_TOOLS = ROOT / 'tools'

for folder in [DIR_METADATA, DIR_SRA, DIR_FASTQ, DIR_TEMP, DIR_TOOLS]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'BioProject : {BIOPROJECT}')
print('Mode analisis:', 'UJI (1 run)' if TEST_MODE else 'LENGKAP (26 run)')
print(f'Folder hasil: {ROOT.resolve()}')


## 2. Mengambil metadata dari NCBI

NCBI RunInfo digunakan untuk memperoleh accession `SRR`, tipe layout, platform, BioSample, dan informasi dasar lainnya. Metadata disimpan agar tahap ini tidak perlu diulang saat notebook dijalankan kembali.


In [ ]:
# 3. Mengambil tabel RunInfo BioProject
runinfo_file = DIR_METADATA / f'{BIOPROJECT}_RunInfo.csv'
eutils_url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'

if runinfo_file.exists():
    metadata = pd.read_csv(runinfo_file, dtype=str)
    print('[OK] Tahap selesai.')
else:
    search = requests.get(
        f'{eutils_url}/esearch.fcgi',
        params={'db': 'sra', 'term': BIOPROJECT, 'usehistory': 'y',
                'retmax': 0, 'retmode': 'json'},
        timeout=120, headers={'User-Agent': 'IPB-microbiome-practicum/1.0'}
    )
    search.raise_for_status()
    search_result = search.json()['esearchresult']
    if int(search_result['count']) == 0:
        raise RuntimeError(f'Tidak ada run SRA untuk {BIOPROJECT}.')
    response = requests.get(
        f'{eutils_url}/efetch.fcgi',
        params={'db': 'sra', 'query_key': search_result['querykey'],
                'WebEnv': search_result['webenv'], 'rettype': 'runinfo',
                'retmode': 'text'},
        timeout=120, headers={'User-Agent': 'IPB-microbiome-practicum/1.0'}
    )
    response.raise_for_status()
    metadata = pd.read_csv(StringIO(response.text), dtype=str)
    metadata = metadata.dropna(how='all')
    if metadata.empty or 'Run' not in metadata.columns:
        raise RuntimeError('NCBI tidak mengembalikan daftar Run. Periksa accession BioProject dan koneksi internet.')
    metadata.to_csv(runinfo_file, index=False)
    print('[OK] Tahap selesai.')

print(f'Jumlah run ditemukan: {len(metadata)}')
kolom = [c for c in ['Run', 'BioSample', 'LibraryLayout', 'Platform', 'Model', 'spots', 'bases', 'size_MB'] if c in metadata.columns]
display(metadata[kolom].head(10))


## 3. Seleksi dan pengelompokan data seperti pada NCBI Run Selector

Pada modul Galaxy, sampel dipilih menggunakan filter `LibraryLayout` dan `agrochem_addition`, lalu disimpan sebagai empat daftar accession. Sel berikut mengerjakan langkah yang sama secara otomatis berdasarkan metadata BioSample.

Empat kelompok yang diharapkan adalah:

| Kelompok | LibraryLayout | Jumlah run |
|---|---:|---:|
| Conventional farming practice | PAIRED | 12 |
| Organic farming practice | PAIRED | 11 |
| Organic farming practice | SINGLE | 2 |
| Not applicable | PAIRED | 1 |

Notebook akan menampilkan peringatan apabila jumlah yang ditemukan berbeda. Daftar accession disimpan di folder `metadata` sehingga dapat diperiksa seperti berkas hasil unduhan dari Run Selector.


In [ ]:
# 4. Mengambil atribut BioSample dan membentuk empat kelompok seperti pada modul
import xml.etree.ElementTree as ET

atribut_file = DIR_METADATA / f'{BIOPROJECT}_BioSample_attributes.csv'
metadata_kelompok_file = DIR_METADATA / f'{BIOPROJECT}_metadata_kelompok.csv'

if atribut_file.exists():
    atribut_biosample = pd.read_csv(atribut_file, dtype=str)
    print('Metadata atribut BioSample dibaca dari berkas yang sudah tersedia.')
else:
    accession_biosample = metadata['BioSample'].dropna().astype(str).unique().tolist()
    if not accession_biosample:
        raise RuntimeError('Kolom BioSample pada RunInfo tidak berisi accession SAMN.')

    # Cari menggunakan accession SAMN dari RunInfo. Cara ini lebih andal daripada
    # mencari BioProject langsung pada indeks BioSample.
    kueri_accession = ' OR '.join(f'{acc}[Accession]' for acc in accession_biosample)
    pencarian_sampel = requests.post(
        f'{eutils_url}/esearch.fcgi',
        data={'db': 'biosample', 'term': kueri_accession, 'usehistory': 'y',
              'retmax': 0, 'retmode': 'json'},
        timeout=120, headers={'User-Agent': 'IPB-microbiome-practicum/1.0'}
    )
    pencarian_sampel.raise_for_status()
    hasil_pencarian = pencarian_sampel.json()['esearchresult']
    if int(hasil_pencarian['count']) == 0:
        raise RuntimeError(
            'Accession BioSample dari RunInfo tidak ditemukan di NCBI. '
            'Periksa koneksi internet atau isi kolom BioSample.'
        )

    respons_sampel = requests.get(
        f'{eutils_url}/efetch.fcgi',
        params={'db': 'biosample', 'query_key': hasil_pencarian['querykey'],
                'WebEnv': hasil_pencarian['webenv'], 'retmode': 'xml'},
        timeout=180, headers={'User-Agent': 'IPB-microbiome-practicum/1.0'}
    )
    respons_sampel.raise_for_status()
    akar_xml = ET.fromstring(respons_sampel.content)

    baris_atribut = []
    for sampel in akar_xml.findall('.//BioSample'):
        baris = {'BioSample': sampel.attrib.get('accession', '')}
        for atribut in sampel.findall('./Attributes/Attribute'):
            nama = atribut.attrib.get('harmonized_name') or atribut.attrib.get('attribute_name')
            if nama:
                baris[nama.strip()] = (atribut.text or '').strip()
        baris_atribut.append(baris)

    atribut_biosample = pd.DataFrame(baris_atribut)
    atribut_biosample.to_csv(atribut_file, index=False)
    print('Metadata atribut BioSample berhasil diambil dari NCBI.')

def normalisasi_nama_kolom(nama):
    return ''.join(ch.lower() for ch in str(nama) if ch.isalnum())

kandidat_agrochem = {
    'agrochemaddition', 'agrochemicaladdition', 'agrochemicaladditions'
}
kolom_agrochem = next(
    (kolom for kolom in atribut_biosample.columns
     if normalisasi_nama_kolom(kolom) in kandidat_agrochem),
    None
)
if kolom_agrochem is None:
    raise RuntimeError(
        'Kolom agrochem_addition tidak ditemukan pada metadata BioSample. '
        'Periksa berkas atribut sebelum melanjutkan.'
    )

atribut_ringkas = atribut_biosample[['BioSample', kolom_agrochem]].copy()
atribut_ringkas = atribut_ringkas.rename(columns={kolom_agrochem: 'agrochem_addition'})
metadata_kelompok = metadata.merge(atribut_ringkas, on='BioSample', how='left')
metadata_kelompok['LibraryLayout'] = metadata_kelompok['LibraryLayout'].str.upper().str.strip()

def tentukan_kelompok(perlakuan, layout):
    perlakuan = str(perlakuan).lower().strip()
    layout = str(layout).upper().strip()
    if 'conventional' in perlakuan and layout == 'PAIRED':
        return 'conventional_paired'
    if 'organic' in perlakuan and layout == 'PAIRED':
        return 'organic_paired'
    if 'organic' in perlakuan and layout == 'SINGLE':
        return 'organic_single'
    if ('not applicable' in perlakuan or perlakuan in {'n/a', 'na'}) and layout == 'PAIRED':
        return 'not_applicable_paired'
    return 'belum_terkelompok'

metadata_kelompok['kelompok_modul'] = metadata_kelompok.apply(
    lambda baris: tentukan_kelompok(
        baris.get('agrochem_addition', ''), baris.get('LibraryLayout', '')
    ),
    axis=1
)

nama_berkas = {
    'conventional_paired': 'SRA_conventional_paired.txt',
    'organic_paired': 'SRA_organic_paired.txt',
    'organic_single': 'SRA_organic_single.txt',
    'not_applicable_paired': 'SRA_not_applicable_paired.txt',
}
jumlah_harapan = {
    'conventional_paired': 12,
    'organic_paired': 11,
    'organic_single': 2,
    'not_applicable_paired': 1,
}

ringkasan = []
for kelompok, berkas in nama_berkas.items():
    accession = metadata_kelompok.loc[
        metadata_kelompok['kelompok_modul'] == kelompok, 'Run'
    ].dropna().astype(str).tolist()
    (DIR_METADATA / berkas).write_text('\n'.join(accession) + '\n', encoding='utf-8')
    ringkasan.append({
        'kelompok': kelompok,
        'ditemukan': len(accession),
        'diharapkan': jumlah_harapan[kelompok],
        'status': 'sesuai' if len(accession) == jumlah_harapan[kelompok] else 'periksa',
        'berkas': berkas,
    })

metadata_kelompok.to_csv(metadata_kelompok_file, index=False)
ringkasan_kelompok = pd.DataFrame(ringkasan)
display(ringkasan_kelompok)

belum_terkelompok = metadata_kelompok[
    metadata_kelompok['kelompok_modul'] == 'belum_terkelompok'
]
if not belum_terkelompok.empty:
    print(f'PERINGATAN: {len(belum_terkelompok)} run belum masuk ke empat kelompok modul.')
    display(belum_terkelompok[[
        c for c in ['Run', 'BioSample', 'agrochem_addition', 'LibraryLayout']
        if c in belum_terkelompok.columns
    ]])

if (ringkasan_kelompok['status'] == 'sesuai').all() and belum_terkelompok.empty:
    print('Semua kelompok dan jumlah run sesuai dengan modul.')
else:
    print('Periksa kelompok berstatus "periksa" sebelum melanjutkan analisis lengkap.')

# Tahap berikutnya memakai metadata yang sudah diperkaya dan dikelompokkan.
metadata = metadata_kelompok


## 4. Memilih run untuk percobaan

> **Checkpoint:** mode lengkap harus menampilkan 26 run sebelum unduhan dimulai.


Mode uji memilih run berukuran paling kecil agar proses pemeriksaan lebih cepat. Pilihan ini hanya untuk menguji bahwa unduhan dan konversi FASTQ bekerja, bukan untuk analisis akhir.


In [ ]:
# 5. Memilih accession yang akan diunduh
pilihan = metadata.copy()
if 'size_MB' in pilihan.columns:
    pilihan['_size'] = pd.to_numeric(pilihan['size_MB'], errors='coerce')
    pilihan = pilihan.sort_values('_size', na_position='last')

if TEST_MODE:
    pilihan = pilihan.head(MAX_RUNS)

RUNS = pilihan['Run'].dropna().astype(str).tolist()
if not RUNS:
    raise RuntimeError('Tidak ada accession SRR yang dapat dipilih.')

RUN_SIZE_MB = dict(zip(pilihan['Run'], pd.to_numeric(pilihan['size_MB'], errors='coerce'))) if 'size_MB' in pilihan.columns else {}
TOTAL_SIZE_MB = sum(v for v in RUN_SIZE_MB.values() if pd.notna(v))
print(f'Jumlah run yang akan diproses: {len(RUNS)}')
if TOTAL_SIZE_MB:
    print(f'Perkiraan total arsip SRA: {TOTAL_SIZE_MB:,.2f} MB')
display(pilihan[[c for c in ['Run', 'BioSample', 'LibraryLayout', 'size_MB'] if c in pilihan.columns]])


## 5. Menyiapkan NCBI SRA Toolkit

Sel berikut mencari SRA Toolkit pada komputer. Jika belum tersedia, paket resmi NCBI akan diunduh dan diekstrak ke dalam folder proyek. Tidak diperlukan instalasi sistem atau hak administrator.


In [ ]:
# 6. Mencari atau mengunduh SRA Toolkit resmi NCBI
def find_tool(name):
    found = shutil.which(name)
    if found:
        return Path(found)
    suffix = '.exe' if SYSTEM == 'Windows' else ''
    # rglob juga dapat menemukan folder bernama sama (misalnya tools/mothur).
    # Hanya berkas yang boleh diperlakukan sebagai executable.
    matches = [path for path in DIR_TOOLS.rglob(name + suffix) if path.is_file()]
    if not matches:
        return None
    # Utamakan executable yang nama berkasnya tepat sama dengan nama alat.
    matches.sort(key=lambda path: (path.name.lower() != (name + suffix).lower(), len(path.parts)))
    return matches[0]

def download_file_with_progress(url, destination, label):
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length', 0))
        with destination.open('wb') as handle, tqdm(
            total=total, unit='B', unit_scale=True, unit_divisor=1024, desc=label
        ) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
                    bar.update(len(chunk))

def install_sra_toolkit():
    if SYSTEM == 'Windows':
        url = 'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-win64.zip'
        archive = DIR_TOOLS / 'sratoolkit-win64.zip'
    elif SYSTEM == 'Linux':
        url = 'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz'
        archive = DIR_TOOLS / 'sratoolkit-linux64.tar.gz'
    else:
        raise RuntimeError(f'Instalasi otomatis belum tersedia untuk sistem {SYSTEM}.')

    download_file_with_progress(url, archive, f'SRA Toolkit ({SYSTEM})')
    print('Mengekstrak SRA Toolkit ...')
    if archive.suffix == '.zip':
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(DIR_TOOLS)
    else:
        with tarfile.open(archive, 'r:gz') as tf:
            tf.extractall(DIR_TOOLS)
    archive.unlink(missing_ok=True)

PREFETCH = find_tool('prefetch')
FASTERQ = find_tool('fasterq-dump')
if not PREFETCH or not FASTERQ:
    install_sra_toolkit()
    PREFETCH = find_tool('prefetch')
    FASTERQ = find_tool('fasterq-dump')

if not PREFETCH or not FASTERQ:
    raise RuntimeError('SRA Toolkit tidak ditemukan setelah instalasi.')

print('[OK] Tahap selesai.')
print('[OK] Tahap selesai.')
version = subprocess.run([str(FASTERQ), '--version'], capture_output=True, text=True, check=True)
print(version.stdout.strip())


## 6. Mengunduh SRA dan membuat FASTQ

`prefetch` mengambil arsip run dari NCBI. `fasterq-dump` kemudian membuat berkas FASTQ. Untuk data paired-end, hasilnya berupa `_1.fastq.gz` dan `_2.fastq.gz`; untuk single-end, hasilnya satu berkas `.fastq.gz`.

> Proses ini memerlukan internet dan ruang kosong. Jangan menutup notebook selama status run masih menunjukkan *diproses*.


In [ ]:
# Fungsi unduhan, konversi, dan kompresi
def run_command_live(command, label, retries=1, retry_delay=20):
    last_message = ''
    for attempt in range(1, retries + 1):
        suffix = f' (percobaan {attempt}/{retries})' if retries > 1 else ''
        print(f'  {label}{suffix}')
        start = time.time()
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True
        )
        recent = []
        for line in iter(process.stdout.readline, ''):
            line = line.rstrip()
            if line:
                print('   ', line, flush=True)
                recent.append(line)
                recent = recent[-20:]
        process.stdout.close()
        return_code = process.wait()
        if return_code == 0:
            print(f'  [OK] Selesai dalam {(time.time() - start) / 60:.1f} menit')
            return

        last_message = '\n'.join(recent)
        if attempt < retries:
            wait_seconds = retry_delay * attempt
            print(f'  Koneksi gagal. Mencoba kembali dalam {wait_seconds} detik ...')
            time.sleep(wait_seconds)

    raise RuntimeError(f'{label} gagal setelah {retries} percobaan.\n{last_message}')


def gzip_file(source):
    target = source.with_suffix(source.suffix + '.gz')
    if target.exists():
        source.unlink(missing_ok=True)
        return target
    total = source.stat().st_size
    with source.open('rb') as src, gzip.open(target, 'wb', compresslevel=6) as dst, tqdm(
        total=total, unit='B', unit_scale=True, unit_divisor=1024,
        desc=f'Kompresi {source.name}'
    ) as bar:
        while True:
            chunk = src.read(1024 * 1024)
            if not chunk:
                break
            dst.write(chunk)
            bar.update(len(chunk))
    source.unlink()
    return target


def download_run(run):
    existing = sorted(DIR_FASTQ.glob(f'{run}*.fastq.gz'))
    if existing:
        print(f'[OK] {run}: FASTQ sudah tersedia; unduhan dilewati.')
        return existing

    expected = RUN_SIZE_MB.get(run)
    size_text = f' (perkiraan arsip {expected:,.2f} MB)' if expected is not None and pd.notna(expected) else ''
    print(f'\nMemproses {run}{size_text}')
    run_command_live(
        [str(PREFETCH), run, '--output-directory', str(DIR_SRA),
         '--max-size', 'u', '--progress'],
        f'{run}: mengunduh arsip SRA', retries=3, retry_delay=20
    )

    sra_candidates = list((DIR_SRA / run).rglob('*.sra'))
    sra_input = sra_candidates[0] if sra_candidates else run
    temp_run = DIR_TEMP / run
    temp_run.mkdir(parents=True, exist_ok=True)
    run_command_live(
        [str(FASTERQ), str(sra_input), '--split-files', '--threads', str(THREADS),
         '--outdir', str(DIR_FASTQ), '--temp', str(temp_run), '--progress'],
        f'{run}: mengubah SRA menjadi FASTQ', retries=2, retry_delay=10
    )

    outputs = sorted(DIR_FASTQ.glob(f'{run}*.fastq'))
    if not outputs:
        raise RuntimeError(f'{run}: konversi selesai tetapi berkas FASTQ tidak ditemukan.')
    compressed = [gzip_file(path) for path in outputs]
    shutil.rmtree(temp_run, ignore_errors=True)
    total_fastq = sum(path.stat().st_size for path in compressed) / 1024**2
    print(f'[OK] {run}: {len(compressed)} FASTQ.gz ({total_fastq:,.2f} MB) berhasil dibuat')
    return compressed


hasil_fastq = {}
status_unduhan = []
for run in tqdm(RUNS, total=len(RUNS), desc='Progres seluruh run', unit='run'):
    try:
        files = download_run(run)
        hasil_fastq[run] = files
        status_unduhan.append({'run': run, 'status': 'berhasil', 'pesan': ''})
    except Exception as exc:
        hasil_fastq[run] = []
        status_unduhan.append({'run': run, 'status': 'gagal', 'pesan': str(exc)})
        print(f'[GAGAL] {run}: {exc}')

manifest_unduhan = pd.DataFrame(status_unduhan)
manifest_unduhan.to_csv(DIR_METADATA / 'download_manifest.csv', index=False)
display(manifest_unduhan)

gagal = manifest_unduhan[manifest_unduhan['status'] == 'gagal']
if gagal.empty:
    print(f'[OK] Seluruh {len(RUNS)} run berhasil disiapkan.')
else:
    print(f'{len(gagal)} run belum berhasil. Jalankan kembali sel ini untuk mencoba ulang:')
    print(', '.join(gagal['run']))


## 7. Memeriksa hasil

FASTQ yang benar memiliki empat baris untuk setiap read: nama read, urutan basa, tanda `+`, dan nilai kualitas. Sel berikut memeriksa struktur awal berkas tanpa membaca seluruh isinya ke memori.


In [ ]:
# 8. Memvalidasi FASTQ dan membuat manifest hasil
def inspect_fastq(path):
    with gzip.open(path, 'rt') as handle:
        first = [handle.readline().rstrip() for _ in range(4)]
    valid = (
        len(first) == 4 and first[0].startswith('@') and first[2].startswith('+')
        and len(first[1]) == len(first[3]) and len(first[1]) > 0
    )
    return {
        'file': path.name,
        'size_MB': round(path.stat().st_size / 1024**2, 2),
        'first_read': first[0],
        'read_length': len(first[1]),
        'valid_fastq': valid
    }

records = []
for run, files in hasil_fastq.items():
    for path in files:
        row = {'run': run, **inspect_fastq(path)}
        records.append(row)

manifest = pd.DataFrame(records)
manifest_file = DIR_METADATA / 'fastq_manifest.csv'
manifest.to_csv(manifest_file, index=False)
display(manifest)

if manifest.empty or not manifest['valid_fastq'].all():
    raise RuntimeError('Satu atau lebih berkas tidak lolos pemeriksaan awal FASTQ.')

print('[OK] Tahap selesai.')
print('[OK] Tahap selesai.')
print('[OK] Tahap selesai.')


## Hasil yang diharapkan

Tahap ini dinyatakan berhasil jika:

- metadata BioProject tampil sebagai tabel;
- setidaknya satu accession SRR terpilih;
- berkas `.fastq.gz` terbentuk;
- kolom `valid_fastq` bernilai `True`; dan
- pesan **TAHAP PENGAMBILAN DATA BERHASIL** muncul.

Jangan mengaktifkan unduhan seluruh run sebelum mode uji ini selesai tanpa galat.


---

## 8. Menyiapkan FASTA untuk data paired-end dan single-end

Tahap ini menangani **seluruh kelompok data**. Untuk run `PAIRED`, Mothur **Make.contigs** menggabungkan read forward dan reverse menjadi `trim.contigs.fasta`. Untuk run `SINGLE`, satu berkas FASTQ dikonversi langsung menjadi FASTA karena tidak mempunyai pasangan yang dapat digabungkan.

Kedua jalur menghasilkan satu `analysis_fasta` per run. Dengan demikian 12 run konvensional paired, 11 run organik paired, 2 run organik single, dan 1 run kontrol paired semuanya diteruskan ke `Summary.seqs`, `Screen.seqs`, `Unique.seqs`, Kraken2, dan Bracken.

Sel aman dijalankan ulang: keluaran yang sudah tersedia akan digunakan kembali.


In [ ]:
# 9. Menyiapkan Mothur untuk menjalankan Make.contigs
DIR_CONTIGS = ROOT / 'make_contigs'
DIR_CONTIGS.mkdir(parents=True, exist_ok=True)

def install_mothur():
    api_url = 'https://api.github.com/repos/mothur/mothur/releases/latest'
    response = requests.get(
        api_url, timeout=120,
        headers={'Accept': 'application/vnd.github+json',
                 'User-Agent': 'IPB-microbiome-practicum/1.0'}
    )
    response.raise_for_status()
    release = response.json()
    assets = release.get('assets', [])
    machine = platform.machine().lower()

    def suitable(asset):
        name = asset['name'].lower()
        if not name.endswith('.zip') or 'tools' in name or 'noreadline' in name:
            return False
        if SYSTEM == 'Windows':
            return 'win' in name
        if SYSTEM == 'Linux':
            architecture = 'arm64' if machine in {'arm64', 'aarch64'} else 'x86_64'
            return architecture in name and ('ubuntu' in name or 'linux' in name)
        return False

    candidates = [asset for asset in assets if suitable(asset)]
    if not candidates:
        available = ', '.join(asset.get('name', '') for asset in assets)
        raise RuntimeError(
            f'Paket Mothur untuk {SYSTEM} ({machine}) tidak ditemukan. '
            f'Aset yang tersedia: {available}'
        )

    # Pada Colab utamakan paket Ubuntu; pada Linux lokal utamakan paket Linux umum.
    if SYSTEM == 'Linux':
        preferred = 'ubuntu' if IS_COLAB else 'linux'
        candidates.sort(key=lambda asset: preferred not in asset['name'].lower())

    asset = candidates[0]
    archive = DIR_TOOLS / asset['name']
    print(f'Mengunduh Mothur {release.get("tag_name", "")} untuk {SYSTEM} ...')
    download_file_with_progress(asset['browser_download_url'], archive, 'Mothur')
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(DIR_TOOLS / 'mothur')
    archive.unlink(missing_ok=True)

MOTHUR = find_tool('mothur')
if not MOTHUR:
    install_mothur()
    MOTHUR = find_tool('mothur')
if not MOTHUR:
    raise RuntimeError('Mothur tidak ditemukan setelah instalasi.')

if SYSTEM != 'Windows':
    MOTHUR.chmod(MOTHUR.stat().st_mode | 0o111)

print(f'Mothur: {MOTHUR}')
version = subprocess.run(
    [str(MOTHUR), '#get.current()'], capture_output=True, text=True, check=True
)
print('Mothur siap digunakan.')


In [ ]:
# 10. Menyiapkan FASTA seluruh run: Make.contigs untuk PAIRED, konversi untuk SINGLE
def mothur_value(path):
    return f'"{Path(path).resolve().as_posix()}"'

def fastq_gz_to_fasta(fastq_path, fasta_path):
    fasta_path.parent.mkdir(parents=True, exist_ok=True)
    if fasta_path.exists() and fasta_path.stat().st_size > 0:
        return fasta_path
    temporary = fasta_path.with_suffix('.fasta.part')
    read_count = 0
    with gzip.open(fastq_path, 'rt') as source, temporary.open('wt', encoding='utf-8') as target:
        while True:
            header = source.readline().rstrip()
            if not header:
                break
            sequence = source.readline().rstrip()
            plus = source.readline().rstrip()
            quality = source.readline().rstrip()
            if not header.startswith('@') or not plus.startswith('+') or len(sequence) != len(quality):
                raise RuntimeError(f'{fastq_path.name}: struktur FASTQ tidak valid pada read {read_count + 1}.')
            target.write(f'>{header[1:].split()[0]}\n{sequence}\n')
            read_count += 1
    temporary.replace(fasta_path)
    print(f'{fastq_path.name}: {read_count} read dikonversi menjadi FASTA.')
    return fasta_path

def prepare_fasta_run(run):
    run_info = metadata.loc[metadata['Run'].astype(str) == str(run)]
    if run_info.empty:
        raise RuntimeError(f'{run}: metadata tidak ditemukan.')
    layout = str(run_info['LibraryLayout'].iloc[0]).upper().strip()
    group = str(run_info['kelompok_modul'].iloc[0]) if 'kelompok_modul' in run_info else 'tidak_diketahui'
    run_output = DIR_CONTIGS / run
    run_output.mkdir(parents=True, exist_ok=True)

    if layout == 'SINGLE':
        candidates = [DIR_FASTQ / f'{run}.fastq.gz', DIR_FASTQ / f'{run}_1.fastq.gz']
        single_fastq = next((path for path in candidates if path.exists()), None)
        if single_fastq is None:
            matches = sorted(DIR_FASTQ.glob(f'{run}*.fastq.gz'))
            single_fastq = matches[0] if len(matches) == 1 else None
        if single_fastq is None:
            raise FileNotFoundError(f'{run}: satu FASTQ single-end tidak ditemukan.')
        analysis_fasta = run_output / f'{run}.single.fasta'
        fastq_gz_to_fasta(single_fastq, analysis_fasta)
        return {
            'run': run, 'kelompok': group, 'layout': layout,
            'jalur': 'FASTQ ke FASTA', 'status': 'berhasil',
            'analysis_fasta': str(analysis_fasta),
            'trim_contigs': str(analysis_fasta),
            'fasta_size_MB': round(analysis_fasta.stat().st_size / 1024**2, 2),
            'scrap_contigs': None, 'contigs_report': None,
        }

    if layout != 'PAIRED':
        raise RuntimeError(f'{run}: LibraryLayout tidak dikenali: {layout}')
    forward = DIR_FASTQ / f'{run}_1.fastq.gz'
    reverse = DIR_FASTQ / f'{run}_2.fastq.gz'
    if not forward.exists() or not reverse.exists():
        raise FileNotFoundError(f'{run}: pasangan {forward.name} dan {reverse.name} tidak lengkap.')

    def find_output(patterns):
        matches=[]
        for pattern in patterns: matches.extend(run_output.glob(pattern))
        return sorted(set(matches))[0] if matches else None

    trim=find_output(['*.trim.contigs.fasta'])
    scrap=find_output(['*.scrap.contigs.fasta'])
    report=find_output(['*.contigs.report','*.contigs_report'])
    if trim and report:
        print(f'{run}: keluaran Make.contigs sudah tersedia, tahap dilewati.')
    else:
        command=(f'#set.dir(output={mothur_value(run_output)}); '
                 f'make.contigs(ffastq={mothur_value(forward)}, '
                 f'rfastq={mothur_value(reverse)}, processors={THREADS})')
        run_command_live([str(MOTHUR),command],f'{run}: Make.contigs')
        trim=find_output(['*.trim.contigs.fasta'])
        scrap=find_output(['*.scrap.contigs.fasta'])
        report=find_output(['*.contigs.report','*.contigs_report'])
    if trim is None or report is None:
        raise RuntimeError(f'{run}: keluaran Make.contigs tidak lengkap.')
    return {
        'run':run,'kelompok':group,'layout':layout,'jalur':'Make.contigs','status':'berhasil',
        'analysis_fasta':str(trim),'trim_contigs':str(trim),
        'fasta_size_MB':round(trim.stat().st_size/1024**2,2),
        'scrap_contigs':str(scrap) if scrap else None,
        'contigs_report':str(report),
    }

hasil_contigs=[]
for run in tqdm(RUNS,total=len(RUNS),desc='Menyiapkan FASTA seluruh run',unit='run'):
    hasil_contigs.append(prepare_fasta_run(run))
manifest_contigs=pd.DataFrame(hasil_contigs)
manifest_contigs_file=DIR_METADATA/'prepared_fasta_manifest.csv'
manifest_contigs.to_csv(manifest_contigs_file,index=False)
display(manifest_contigs[['run','kelompok','layout','jalur','status','fasta_size_MB']])

expected={'conventional_paired':12,'organic_paired':11,'organic_single':2,'not_applicable_paired':1}
actual=manifest_contigs.groupby('kelompok').size().to_dict()
if not TEST_MODE and actual != expected:
    raise RuntimeError(f'Cakupan run belum lengkap. Ditemukan {actual}, diharapkan {expected}.')
print(f'FASTA siap: {len(manifest_contigs)} run; PAIRED={(manifest_contigs.layout=="PAIRED").sum()}, SINGLE={(manifest_contigs.layout=="SINGLE").sum()}')


---

> **Checkpoint:** jumlah baris manifest harus sama dengan jumlah run yang dipilih.


## 9. Summary.seqs: memeriksa panjang contig

Tahap ini setara dengan alat **Summary.seqs** pada Galaxy. Mothur membaca setiap `analysis_fasta` dan merangkum panjang sekuens pada minimum, persentil, median, rata-rata, serta maksimum.

Perhatikan terutama nilai **97.5%-tile** dan **maximum**. Pada modul Galaxy, informasi tersebut digunakan untuk memilih batas `maxlength` pada tahap **Screen.seqs**. Modul menggunakan nilai `475 bp`; nilai itu belum diterapkan pada tahap ini karena `Summary.seqs` hanya melakukan pemeriksaan.

> Ringkasan satu run dalam mode uji belum mewakili seluruh kelompok. Batas akhir harus diperiksa kembali setelah semua run yang diperlukan selesai diproses.


In [ ]:
# 11. Menjalankan Summary.seqs pada hasil Make.contigs
DIR_SUMMARY = ROOT / 'summary_seqs'
DIR_SUMMARY.mkdir(parents=True, exist_ok=True)

def find_summary_file(folder):
    matches = sorted(folder.glob('*.summary'))
    return matches[0] if matches else None

def summarize_lengths(summary_file):
    table = pd.read_csv(summary_file, sep='\t')
    length_column = next(
        (column for column in table.columns
         if column.lower().replace(' ', '') in {'nbases', 'numofbases'}),
        None
    )
    if length_column is None:
        raise RuntimeError(
            f'Kolom panjang sekuens tidak ditemukan pada {summary_file.name}: '
            f'{list(table.columns)}'
        )
    lengths = pd.to_numeric(table[length_column], errors='coerce').dropna()
    if lengths.empty:
        raise RuntimeError(f'{summary_file.name} tidak berisi panjang sekuens yang valid.')

    quantiles = lengths.quantile(
        [0.00, 0.025, 0.25, 0.50, 0.75, 0.975, 1.00],
        interpolation='nearest'
    )
    return {
        'minimum': int(quantiles.loc[0.00]),
        '2.5%-tile': int(quantiles.loc[0.025]),
        '25%-tile': int(quantiles.loc[0.25]),
        'median': int(quantiles.loc[0.50]),
        '75%-tile': int(quantiles.loc[0.75]),
        '97.5%-tile': int(quantiles.loc[0.975]),
        'maximum': int(quantiles.loc[1.00]),
        'mean': round(float(lengths.mean()), 2),
        'jumlah_sekuens': int(len(lengths)),
    }

def summary_seqs_run(row):
    run = str(row['run'])
    if row['status'] != 'berhasil' or not row.get('analysis_fasta'):
        print(f'{run}: tidak memiliki analysis_fasta; Summary.seqs dilewati.')
        return {'run': run, 'status': 'dilewati'}

    fasta = Path(row['analysis_fasta'])
    run_output = DIR_SUMMARY / run
    run_output.mkdir(parents=True, exist_ok=True)
    summary_file = find_summary_file(run_output)

    if summary_file is not None:
        print(f'{run}: keluaran Summary.seqs sudah tersedia, tahap dilewati.')
    else:
        command = (
            f'#set.dir(output={mothur_value(run_output)}); '
            f'summary.seqs(fasta={mothur_value(fasta)}, processors={THREADS})'
        )
        run_command_live([str(MOTHUR), command], f'{run}: Summary.seqs')
        summary_file = find_summary_file(run_output)

    if summary_file is None:
        raise RuntimeError(f'{run}: berkas keluaran .summary tidak ditemukan.')
    return {
        'run': run,
        'status': 'berhasil',
        'summary_file': str(summary_file),
        **summarize_lengths(summary_file),
    }

hasil_summary = []
for _, row in tqdm(
    manifest_contigs.iterrows(), total=len(manifest_contigs),
    desc='Summary.seqs', unit='run'
):
    hasil_summary.append(summary_seqs_run(row))

ringkasan_summary = pd.DataFrame(hasil_summary)
ringkasan_summary_file = DIR_METADATA / 'summary_seqs_manifest.csv'
ringkasan_summary.to_csv(ringkasan_summary_file, index=False)

kolom_tampil = [
    'run', 'status', 'minimum', '2.5%-tile', '25%-tile', 'median',
    '75%-tile', '97.5%-tile', 'maximum', 'mean', 'jumlah_sekuens'
]
display(ringkasan_summary[[c for c in kolom_tampil if c in ringkasan_summary.columns]])
print(f'Ringkasan tersimpan di: {ringkasan_summary_file.resolve()}')
print('Bandingkan 97.5%-tile dan maximum sebelum menetapkan maxlength pada Screen.seqs.')


---

> **Checkpoint:** periksa jumlah sekuens masuk, dipertahankan, dan dibuang untuk setiap run.


## 10. Screen.seqs: menyaring contig berdasarkan kualitas

Tahap ini setara dengan alat **Screen.seqs** pada Galaxy. Sekuens dipertahankan apabila panjangnya berada dalam batas yang ditentukan dan tidak mengandung basa ambigu (`N`). Parameter dibuat sama dengan modul:

| Parameter | Nilai | Arti |
|---|---:|---|
| `minlength` | 10 | Membuang sekuens yang lebih pendek dari 10 bp |
| `maxlength` | 475 | Membuang sekuens yang lebih panjang dari 475 bp |
| `maxambig` | 0 | Membuang sekuens yang memiliki satu atau lebih basa ambigu |

Keluaran utama adalah `*.good.fasta`, yaitu sekuens yang lolos dan digunakan pada tahap berikutnya. Nama sekuens yang tidak lolos disimpan pada `*.bad.accnos` agar proses penyaringan dapat ditelusuri.


In [ ]:
# 12. Menjalankan Screen.seqs dengan parameter yang sama seperti modul Galaxy
MIN_LENGTH = 10
MAX_LENGTH = 475
MAX_AMBIG = 0

DIR_SCREEN = ROOT / 'screen_seqs'
DIR_SCREEN.mkdir(parents=True, exist_ok=True)

def find_first(folder, patterns):
    matches = []
    for pattern in patterns:
        matches.extend(folder.glob(pattern))
    return sorted(set(matches))[0] if matches else None

def count_fasta(path):
    count = 0
    with Path(path).open('rt', encoding='utf-8', errors='replace') as handle:
        for line in handle:
            if line.startswith('>'):
                count += 1
    return count

def count_nonempty_lines(path):
    if path is None or not Path(path).exists():
        return 0
    with Path(path).open('rt', encoding='utf-8', errors='replace') as handle:
        return sum(1 for line in handle if line.strip())

def screen_seqs_run(row):
    run = str(row['run'])
    if row['status'] != 'berhasil' or not row.get('analysis_fasta'):
        print(f'{run}: tidak memiliki analysis_fasta; Screen.seqs dilewati.')
        return {'run': run, 'status': 'dilewati'}

    input_fasta = Path(row['analysis_fasta'])
    run_output = DIR_SCREEN / run
    run_output.mkdir(parents=True, exist_ok=True)
    good_fasta = find_first(run_output, ['*.good.fasta'])
    bad_accnos = find_first(run_output, ['*.bad.accnos'])

    if good_fasta is not None and bad_accnos is not None:
        print(f'{run}: keluaran Screen.seqs sudah tersedia, tahap dilewati.')
    else:
        command = (
            f'#set.dir(output={mothur_value(run_output)}); '
            f'screen.seqs(fasta={mothur_value(input_fasta)}, '
            f'minlength={MIN_LENGTH}, maxlength={MAX_LENGTH}, '
            f'maxambig={MAX_AMBIG}, processors={THREADS})'
        )
        run_command_live([str(MOTHUR), command], f'{run}: Screen.seqs')
        good_fasta = find_first(run_output, ['*.good.fasta'])
        bad_accnos = find_first(run_output, ['*.bad.accnos'])

    if good_fasta is None:
        raise RuntimeError(f'{run}: good.fasta tidak ditemukan.')

    before = count_fasta(input_fasta)
    retained = count_fasta(good_fasta)
    rejected = count_nonempty_lines(bad_accnos)
    retention = round(100 * retained / before, 2) if before else 0.0
    return {
        'run': run,
        'kelompok': row.get('kelompok'),
        'layout': row.get('layout'),
        'status': 'berhasil',
        'sekuens_awal': before,
        'dipertahankan': retained,
        'dibuang': rejected,
        'retensi_persen': retention,
        'good_fasta': str(good_fasta),
        'bad_accnos': str(bad_accnos) if bad_accnos else None,
    }

hasil_screen = []
for _, row in tqdm(
    manifest_contigs.iterrows(), total=len(manifest_contigs),
    desc='Screen.seqs', unit='run'
):
    hasil_screen.append(screen_seqs_run(row))

manifest_screen = pd.DataFrame(hasil_screen)
manifest_screen_file = DIR_METADATA / 'screen_seqs_manifest.csv'
manifest_screen.to_csv(manifest_screen_file, index=False)

kolom_tampil = [
    'run', 'status', 'sekuens_awal', 'dipertahankan',
    'dibuang', 'retensi_persen'
]
display(manifest_screen[[c for c in kolom_tampil if c in manifest_screen.columns]])
print(f'Parameter: minlength={MIN_LENGTH}, maxlength={MAX_LENGTH}, maxambig={MAX_AMBIG}')
print(f'Manifest tersimpan di: {manifest_screen_file.resolve()}')


---

## 11. Unique.seqs: menggabungkan sekuens yang identik

Tahap ini setara dengan alat **Unique.seqs** pada Galaxy. Banyak read dapat mempunyai urutan basa yang persis sama. Mothur menyimpan satu salinan sebagai sekuens perwakilan dan mencatat banyaknya kemunculan sekuens tersebut di dalam `count_table`.

Keluaran utamanya adalah:

- `*.unique.fasta`: satu salinan untuk setiap sekuens yang unik; dan
- `*.count_table`: jumlah read yang diwakili oleh setiap sekuens unik.

Jumlah baris pada FASTA akan berkurang, tetapi jumlah total pada `count_table` harus tetap sama dengan jumlah sekuens yang masuk dari `good.fasta`. Artinya, data kelimpahan tidak dibuang; sekuens identik hanya diringkas agar tahap selanjutnya lebih efisien.


In [ ]:
# 13. Menjalankan Unique.seqs dan membuat count_table seperti pada Galaxy
DIR_UNIQUE = ROOT / 'unique_seqs'
DIR_UNIQUE.mkdir(parents=True, exist_ok=True)

def unique_seqs_run(row):
    run = str(row['run'])
    if row['status'] != 'berhasil' or not row.get('good_fasta'):
        print(f'{run}: tidak memiliki good.fasta; Unique.seqs dilewati.')
        return {'run': run, 'status': 'dilewati'}

    input_fasta = Path(row['good_fasta'])
    run_output = DIR_UNIQUE / run
    run_output.mkdir(parents=True, exist_ok=True)
    unique_fasta = find_first(run_output, ['*.unique.fasta'])
    count_table = find_first(run_output, ['*.count_table'])

    if unique_fasta is not None and count_table is not None:
        print(f'{run}: keluaran Unique.seqs sudah tersedia, tahap dilewati.')
    else:
        command = (
            f'#set.dir(output={mothur_value(run_output)}); '
            f'unique.seqs(fasta={mothur_value(input_fasta)}, format=count)'
        )
        run_command_live([str(MOTHUR), command], f'{run}: Unique.seqs')
        unique_fasta = find_first(run_output, ['*.unique.fasta'])
        count_table = find_first(run_output, ['*.count_table'])

    if unique_fasta is None or count_table is None:
        raise RuntimeError(f'{run}: unique.fasta atau count_table tidak ditemukan.')

    count_data = pd.read_csv(count_table, sep='\t')
    total_column = next(
        (column for column in count_data.columns if column.lower() == 'total'),
        None
    )
    if total_column is None:
        raise RuntimeError(f'{run}: kolom total tidak ditemukan pada count_table.')

    input_count = count_fasta(input_fasta)
    unique_count = count_fasta(unique_fasta)
    represented_count = int(pd.to_numeric(count_data[total_column], errors='coerce').sum())
    if represented_count != input_count:
        raise RuntimeError(
            f'{run}: total count_table ({represented_count}) tidak sama dengan '
            f'jumlah good.fasta ({input_count}).'
        )

    duplicates = input_count - unique_count
    reduction = round(100 * duplicates / input_count, 2) if input_count else 0.0
    return {
        'run': run,
        'kelompok': row.get('kelompok'),
        'layout': row.get('layout'),
        'status': 'berhasil',
        'sekuens_masuk': input_count,
        'sekuens_unik': unique_count,
        'duplikat_diringkas': duplicates,
        'reduksi_persen': reduction,
        'total_count_table': represented_count,
        'unique_fasta': str(unique_fasta),
        'count_table': str(count_table),
    }

hasil_unique = []
for _, row in tqdm(
    manifest_screen.iterrows(), total=len(manifest_screen),
    desc='Unique.seqs', unit='run'
):
    hasil_unique.append(unique_seqs_run(row))

manifest_unique = pd.DataFrame(hasil_unique)
manifest_unique_file = DIR_METADATA / 'unique_seqs_manifest.csv'
manifest_unique.to_csv(manifest_unique_file, index=False)

kolom_tampil = [
    'run', 'status', 'sekuens_masuk', 'sekuens_unik',
    'duplikat_diringkas', 'reduksi_persen', 'total_count_table'
]
display(manifest_unique[[c for c in kolom_tampil if c in manifest_unique.columns]])
print(f'Manifest tersimpan di: {manifest_unique_file.resolve()}')


---

> **Keamanan:** API key diminta melalui `getpass` dan tidak disimpan di notebook.


## 12A. Menghubungkan notebook ke Galaxy Europe

Jalur ini direkomendasikan ketika komputer lokal atau Google Colab tidak cukup untuk menyimpan database PlusPFP. Notebook tetap menjadi pengendali, sedangkan Kraken2 dijalankan pada server Galaxy Europe.

### Membuat API key

1. Masuk ke [Galaxy Europe](https://usegalaxy.eu).
2. Buka **User** atau ikon akun, lalu pilih **Preferences**.
3. Buka **Manage API key**.
4. Buat atau salin API key.
5. Jalankan sel di bawah dan masukkan API key ketika diminta.

API key dimasukkan melalui kotak rahasia dan hanya disimpan di memori sesi. Jangan menuliskannya langsung di dalam kode, menyimpan output yang memuat key, atau membagikannya kepada orang lain.

Sel pertama hanya menguji koneksi dan mencari Kraken2. Belum ada data yang diunggah dan belum ada pekerjaan analisis yang dijalankan.


In [ ]:
# Menghubungkan notebook ke Galaxy Australia
import json
import subprocess
import sys
from getpass import getpass
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas'])
    import pandas as pd

# Sel ini dapat dijalankan langsung untuk melanjutkan sesi lama.
if 'ROOT' not in globals():
    try:
        import google.colab  # noqa: F401
        ROOT = Path('/content/mikrobioma_ncbi')
    except ImportError:
        ROOT = Path.cwd() / 'mikrobioma_ncbi'
if 'DIR_METADATA' not in globals():
    DIR_METADATA = ROOT / 'metadata'

try:
    from bioblend.galaxy import GalaxyInstance
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bioblend'])
    from bioblend.galaxy import GalaxyInstance

GALAXY_SERVER = 'australia'
GALAXY_URL = 'https://usegalaxy.org.au'
GALAXY_API_KEY = getpass('Masukkan API key Galaxy Australia (input disembunyikan): ')
if not GALAXY_API_KEY.strip():
    raise RuntimeError('API key belum diisi.')

# Quick resume: hasil lokal tidak dihitung ulang. Manifest Screen.seqs dimuat
# kembali jika kernel pernah dihentikan tetapi folder kerja masih tersedia.
if 'manifest_screen' not in globals():
    resume_file = DIR_METADATA / 'screen_seqs_manifest.csv'
    if not resume_file.exists():
        raise RuntimeError(
            'Manifest Screen.seqs tidak ditemukan. Jika runtime Colab sudah '
            'dihapus, folder /content juga ikut hilang dan hasil perlu dipulihkan '
            'dari Google Drive/backup.'
        )
    manifest_screen = pd.read_csv(resume_file)
    print(f'Melanjutkan dari manifest: {resume_file}')

galaxy = GalaxyInstance(url=GALAXY_URL, key=GALAXY_API_KEY.strip())
try:
    galaxy_version = galaxy.config.get_version()
    galaxy_user = galaxy.users.get_current_user()
except Exception as exc:
    GALAXY_API_KEY = ''
    raise RuntimeError(
        'Koneksi Galaxy Australia gagal. Periksa API key dan koneksi internet. '
        f'Detail: {exc}'
    ) from exc

kraken_tools = galaxy.tools.get_tools(name='Kraken2')
kraken_tools = [
    tool for tool in kraken_tools
    if 'kraken2' in (str(tool.get('id', '')) + ' ' + str(tool.get('name', ''))).lower()
]
if not kraken_tools:
    raise RuntimeError('Koneksi berhasil, tetapi Kraken2 tidak ditemukan di Galaxy Australia.')

tool_table = pd.DataFrame([
    {'name': t.get('name'), 'version': t.get('version'), 'id': t.get('id')}
    for t in kraken_tools
]).drop_duplicates()
email = galaxy_user.get('email', '(tidak ditampilkan)')
masked_email = email[:2] + '***' + email[email.find('@'):] if '@' in email else '(akun terhubung)'
print(f'Galaxy URL     : {GALAXY_URL}')
print(f'Galaxy version : {galaxy_version}')
print(f'Akun           : {masked_email}')
display(tool_table)
print('KONEKSI GALAXY AUSTRALIA BERHASIL')


### Memeriksa parameter Kraken2 pada Galaxy Europe

Galaxy dapat memasang beberapa versi tool secara bersamaan. Sel berikut memilih Kraken2 `2.17.1+galaxy0`, membaca skema formulirnya, dan mencari pilihan database yang mengandung nama `PlusPF` atau `PlusPFP`.

Langkah ini diperlukan karena API menggunakan **nama parameter dan nilai internal**, bukan hanya teks yang terlihat pada formulir. Nilai tersebut harus diperoleh langsung dari Galaxy Europe agar notebook tidak menebak konfigurasi.

Sel ini bersifat baca-saja: belum membuat history, belum mengunggah FASTA, dan belum menjalankan Kraken2.


In [ ]:
# Memilih Kraken2 terbaru dan membaca database pada Galaxy Australia
import re

def version_key(tool):
    return tuple(int(x) for x in re.findall(r'\d+', str(tool.get('version', '0'))))

selected_tool = sorted(kraken_tools, key=version_key, reverse=True)[0]
KRAKEN_GALAXY_VERSION = selected_tool.get('version')
KRAKEN_GALAXY_TOOL_ID = selected_tool['id']
kraken_tool_schema = galaxy.tools.show_tool(KRAKEN_GALAXY_TOOL_ID, io_details=True)

schema_file = DIR_METADATA / 'galaxy_australia_kraken2_tool_schema.json'
schema_file.write_text(json.dumps(kraken_tool_schema, indent=2, ensure_ascii=False), encoding='utf-8')

def find_named_parameter(value, wanted):
    if isinstance(value, dict):
        if value.get('name') == wanted:
            return value
        for child in value.values():
            found = find_named_parameter(child, wanted)
            if found:
                return found
    elif isinstance(value, list):
        for child in value:
            found = find_named_parameter(child, wanted)
            if found:
                return found
    return None

database_parameter = find_named_parameter(kraken_tool_schema, 'kraken2_database')
database_options = []
for option in (database_parameter or {}).get('options', []) or []:
    if isinstance(option, dict):
        label = str(option.get('label', option.get('name', option.get('value', ''))))
        value = option.get('value')
    elif isinstance(option, (list, tuple)) and len(option) >= 2:
        label, value = str(option[0]), option[1]
    else:
        continue
    database_options.append({'label': label, 'value': value})

pluspfp = [row for row in database_options if 'pluspfp' in (row['label'] + ' ' + str(row['value'])).lower()]
if not pluspfp:
    raise RuntimeError(
        'Database PlusPFP tidak tersedia pada Kraken2 Galaxy Australia. '
        'Jangan memilih database lain secara otomatis; lihat tabel tool/database server.'
    )

# Pilih opsi PlusPFP terbaru apabila server menyediakan lebih dari satu versi.
GALAXY_KRAKEN_DB = pluspfp[-1]['value']
print(f'Tool ID  : {KRAKEN_GALAXY_TOOL_ID}')
print(f'Versi    : {KRAKEN_GALAXY_VERSION}')
print(f'Database : {pluspfp[-1]["label"]}')
display(pd.DataFrame(pluspfp))
print(f'Skema tersimpan: {schema_file.resolve()}')


### Menjalankan Kraken2 pada Galaxy Europe

Sel berikut mengotomatisasi langkah yang sebelumnya dilakukan melalui antarmuka Galaxy:

1. membuat atau menggunakan kembali History bernama `Praktikum Mikrobiome - Notebook`;
2. mengunggah `unique.fasta` dari run uji;
3. menunggu unggahan berwarna hijau;
4. menjalankan Kraken2 `2.17.1+galaxy0` dalam mode **Single**;
5. menggunakan `confidence=0.1`, `minimum hit groups=2`, dan database **PlusPFP 2022-06-07**;
6. menunggu job selesai; dan
7. mengunduh klasifikasi dan report ke folder notebook.

Sel aman dijalankan ulang pada sesi yang sama: state disimpan dalam berkas JSON dan dataset/job yang sudah ada akan digunakan kembali. Jangan menutup runtime Colab ketika unggahan atau job masih berjalan.


In [ ]:
# Menjalankan Kraken2 untuk seluruh screened reads melalui Galaxy Australia
GALAXY_HISTORY_NAME = 'Praktikum Mikrobiome - Notebook Australia'
GALAXY_POLL_SECONDS = 15

DIR_GALAXY = ROOT / 'galaxy_australia'
DIR_GALAXY.mkdir(parents=True, exist_ok=True)
galaxy_state_file = DIR_METADATA / 'galaxy_australia_state.json'

def load_galaxy_state():
    if galaxy_state_file.exists():
        try:
            return json.loads(galaxy_state_file.read_text(encoding='utf-8'))
        except (json.JSONDecodeError, OSError):
            pass
    return {}

def save_galaxy_state(state):
    galaxy_state_file.write_text(
        json.dumps(state, indent=2, ensure_ascii=False), encoding='utf-8'
    )

def galaxy_object_exists(object_type, object_id):
    if not object_id:
        return False
    try:
        if object_type == 'history':
            galaxy.histories.show_history(object_id)
        elif object_type == 'dataset':
            galaxy.datasets.show_dataset(object_id)
        elif object_type == 'job':
            galaxy.jobs.show_job(object_id)
        return True
    except Exception:
        return False

def wait_for_dataset(dataset_id, label):
    start = time.time()
    last_state = None
    while True:
        dataset = galaxy.datasets.show_dataset(dataset_id)
        state = dataset.get('state', 'unknown')
        if state != last_state:
            elapsed = (time.time() - start) / 60
            print(f'{label}: {state} ({elapsed:.1f} menit)', flush=True)
            last_state = state
        if state == 'ok':
            return dataset
        if state in {'error', 'discarded', 'failed_metadata', 'deleted'}:
            raise RuntimeError(f'{label} gagal dengan status {state}.')
        time.sleep(GALAXY_POLL_SECONDS)

def wait_for_job(job_id, label):
    start = time.time()
    last_state = None
    while True:
        job = galaxy.jobs.show_job(job_id, full_details=True)
        state = job.get('state', 'unknown')
        if state != last_state:
            elapsed = (time.time() - start) / 60
            print(f'{label}: {state} ({elapsed:.1f} menit)', flush=True)
            last_state = state
        if state == 'ok':
            return job
        if state in {'error', 'deleted', 'deleted_new'}:
            stderr = job.get('stderr', '')
            raise RuntimeError(f'{label} gagal dengan status {state}.\n{stderr[-2000:]}')
        time.sleep(GALAXY_POLL_SECONDS)

state = load_galaxy_state()

# Pilih satu FASTA pada TEST_MODE, atau seluruh FASTA pada analisis lengkap.
valid_screen = manifest_screen[manifest_screen['status'] == 'berhasil'].copy()
if valid_screen.empty:
    raise RuntimeError('Tidak ada good.fasta hasil Screen.seqs yang siap diunggah.')

# Buat atau gunakan kembali History.
history_id = state.get('history_id')
if not galaxy_object_exists('history', history_id):
    matching_histories = galaxy.histories.get_histories(name=GALAXY_HISTORY_NAME)
    if matching_histories:
        history_id = matching_histories[0]['id']
        print(f'Menggunakan History yang sudah ada: {GALAXY_HISTORY_NAME}')
    else:
        history_id = galaxy.histories.create_history(GALAXY_HISTORY_NAME)['id']
        print(f'History baru dibuat: {GALAXY_HISTORY_NAME}')
    state['history_id'] = history_id
    save_galaxy_state(state)

state.setdefault('runs', {})
hasil_galaxy = []

for _, row in valid_screen.iterrows():
    run = str(row['run'])
    fasta = Path(row['good_fasta'])
    run_state = state['runs'].setdefault(run, {})
    print(f'\n=== {run} ===')

    # Upload FASTA hanya jika dataset sebelumnya tidak lagi tersedia.
    dataset_id = run_state.get('input_dataset_id')
    if not galaxy_object_exists('dataset', dataset_id):
        print(f'Mengunggah {fasta.name} ({fasta.stat().st_size / 1024**2:.2f} MB) ...')
        upload_result = galaxy.tools.upload_file(
            str(fasta), history_id, file_name=f'{run}.screened.fasta',
            file_type='fasta', dbkey='?'
        )
        dataset_id = upload_result['outputs'][0]['id']
        run_state['input_dataset_id'] = dataset_id
        save_galaxy_state(state)
    else:
        print('Dataset FASTA sudah ada di Galaxy; upload dilewati.')
    wait_for_dataset(dataset_id, f'{run} upload')

    # Jalankan Kraken2 hanya jika job sebelumnya tidak tersedia.
    job_id = run_state.get('kraken_job_id')
    if not galaxy_object_exists('job', job_id):
        tool_inputs = {
            'single_paired|single_paired_selector': 'no',
            'single_paired|input_sequences': {'src': 'hda', 'id': dataset_id},
            'use_names': False,
            'confidence': 0.1,
            'min_base_quality': 0,
            'minimum_hit_groups': 2,
            'quick': False,
            'split_reads': False,
            'report|create_report': 'true',
            'report|use_mpa_style': False,
            'report|report_zero_counts': False,
            'report|report_minimizer_data': False,
            'kraken2_database': GALAXY_KRAKEN_DB,
        }
        print('Mengirim job Kraken2 ke Galaxy Australia ...')
        # Tool ID sudah memuat versi lengkap. Kompatibel dengan BioBlend lama
        # maupun baru tanpa argumen tool_version.
        invocation = galaxy.tools.run_tool(
            history_id, KRAKEN_GALAXY_TOOL_ID, tool_inputs
        )
        if not invocation.get('jobs'):
            raise RuntimeError(f'Galaxy tidak mengembalikan job Kraken2: {invocation}')
        job_id = invocation['jobs'][0]['id']
        run_state['kraken_job_id'] = job_id
        run_state['outputs'] = {
            output.get('output_name', output.get('name', f'output_{index}')): output['id']
            for index, output in enumerate(invocation.get('outputs', []))
        }
        save_galaxy_state(state)
    else:
        print('Job Kraken2 sudah ada; pengiriman ulang dilewati.')

    wait_for_job(job_id, f'{run} Kraken2')

    downloaded = {}
    for output_name, output_id in run_state.get('outputs', {}).items():
        dataset = wait_for_dataset(output_id, f'{run} {output_name}')
        safe_name = ''.join(ch if ch.isalnum() or ch in '._-' else '_' for ch in output_name)
        destination = DIR_GALAXY / f'{run}.{safe_name}.tsv'
        if not destination.exists() or destination.stat().st_size == 0:
            galaxy.datasets.download_dataset(
                output_id, file_path=str(destination), use_default_filename=False
            )
        downloaded[output_name] = str(destination)
        print(f'Diunduh: {destination.name} ({destination.stat().st_size / 1024:.1f} KB)')

    hasil_galaxy.append({
        'run': run,
        'history_id': history_id,
        'input_dataset_id': dataset_id,
        'job_id': job_id,
        **downloaded,
    })

manifest_galaxy_kraken = pd.DataFrame(hasil_galaxy)
manifest_galaxy_file = DIR_METADATA / 'galaxy_australia_kraken2_manifest.csv'
manifest_galaxy_kraken.to_csv(manifest_galaxy_file, index=False)
display(manifest_galaxy_kraken)
print(f'History: {GALAXY_URL}/histories/view?id={history_id}')
print(f'Manifest: {manifest_galaxy_file.resolve()}')


---

## 13. Bracken dan Krona melalui Galaxy Europe

Kode berikut menyiapkan seluruh tahap setelah Kraken2. Secara bawaan `RUN_DOWNSTREAM_GALAXY = False`, sehingga sel dapat ditinjau tanpa mengirim job baru. Setelah hasil Kraken2 tersedia dan konfigurasi telah diperiksa, ubah nilainya menjadi `True`.

Parameter Bracken mengikuti modul: tingkat **Species**, ambang minimum `10`, keluaran Kraken-style report aktif, dan k-mer distribution **PlusPF (2021-05-17)**. Perhatikan bahwa database Kraken pada modul bernama **PlusPFP 2022**, sedangkan distribusi Bracken bernama **PlusPF 2021**. Notebook akan menampilkan peringatan karena pasangan database idealnya berasal dari build yang sama.


In [ ]:
# 19. Menemukan Bracken/KrakenTools/Krona dan membaca skemanya
RUN_DOWNSTREAM_GALAXY = False

def newest_tool(name_contains, id_contains=None):
    tools = galaxy.tools.get_tools(name=name_contains)
    filtered = []
    for tool in tools:
        haystack = (str(tool.get('name', '')) + ' ' + str(tool.get('id', ''))).lower()
        if name_contains.lower() not in haystack:
            continue
        if id_contains and id_contains.lower() not in str(tool.get('id', '')).lower():
            continue
        filtered.append(tool)
    if not filtered:
        return None
    def version_key(tool):
        return tuple(int(x) for x in __import__('re').findall(r'\d+', str(tool.get('version', '0'))))
    return sorted(filtered, key=version_key, reverse=True)[0]

BRACKEN_TOOL = newest_tool('Bracken', '/bracken/')
KREPORT2KRONA_TOOL = newest_tool('Convert kraken report', 'kreport2krona')
KRONA_TOOL = newest_tool('Krona pie chart')

required_tools = {
    'Bracken': BRACKEN_TOOL,
    'Kraken report -> Krona': KREPORT2KRONA_TOOL,
    'Krona': KRONA_TOOL,
}
tool_rows = []
GALAXY_SCHEMAS = {}
for label, tool in required_tools.items():
    if tool:
        schema = galaxy.tools.show_tool(tool['id'], io_details=True)
        GALAXY_SCHEMAS[label] = schema
        (DIR_METADATA / f'galaxy_{label.lower().replace(" ", "_").replace("/", "_")}_schema.json').write_text(
            json.dumps(schema, indent=2, ensure_ascii=False), encoding='utf-8'
        )
        tool_rows.append({'tahap': label, 'name': tool.get('name'), 'version': tool.get('version'), 'id': tool.get('id')})
    else:
        tool_rows.append({'tahap': label, 'name': None, 'version': None, 'id': None})
display(pd.DataFrame(tool_rows))

if BRACKEN_TOOL is None:
    raise RuntimeError('Tool Bracken tidak ditemukan di Galaxy Europe.')
print('Skema tool downstream berhasil dibaca. Belum ada job yang dikirim.')


In [ ]:
# 20. Helper generik untuk menyusun input tool Galaxy dari label formulir
def schema_parameters(schema):
    records = []
    def walk(items, prefix=''):
        for item in items or []:
            name = item.get('name', '')
            current = f'{prefix}|{name}' if prefix else name
            if item.get('type') == 'conditional':
                test = item.get('test_param', {})
                records.append({'path': f'{current}|{test.get("name")}', **test})
                for case in item.get('cases', []):
                    walk(case.get('inputs', []), current)
            else:
                records.append({'path': current, **item})
    walk(schema.get('inputs', []))
    return records

def option_value(parameter, wanted):
    wanted = wanted.lower()
    for option in parameter.get('options', []) or []:
        if isinstance(option, (list, tuple)) and len(option) >= 2:
            label, value = str(option[0]), option[1]
        elif isinstance(option, dict):
            label = str(option.get('label', option.get('name', '')))
            value = option.get('value')
        else:
            continue
        if wanted in label.lower():
            return value
    return None

def output_map(invocation):
    return {
        output.get('output_name', output.get('name', f'output_{i}')): output['id']
        for i, output in enumerate(invocation.get('outputs', []))
    }

def submit_once(run_state, state_key, tool, inputs, history_id):
    job_id = run_state.get(f'{state_key}_job_id')
    if not galaxy_object_exists('job', job_id):
        result = galaxy.tools.run_tool(history_id, tool['id'], inputs)
        if not result.get('jobs'):
            raise RuntimeError(f'Galaxy tidak membuat job {state_key}: {result}')
        job_id = result['jobs'][0]['id']
        run_state[f'{state_key}_job_id'] = job_id
        run_state[f'{state_key}_outputs'] = output_map(result)
        save_galaxy_state(state)
    wait_for_job(job_id, f'{run} {state_key}')
    return run_state.get(f'{state_key}_outputs', {})

def find_output(outputs, words):
    for name, dataset_id in outputs.items():
        if all(word.lower() in name.lower() for word in words):
            return dataset_id
    return None


In [ ]:
# Menjalankan Bracken dengan distribusi yang kompatibel dengan Kraken2
BRACKEN_LEVEL = 'Species'
BRACKEN_THRESHOLD = 10
BRACKEN_DISTRIBUTION = 'PlusPFP'

if not RUN_DOWNSTREAM_GALAXY:
    print('MODE REVIEW: Bracken belum dijalankan. Ubah RUN_DOWNSTREAM_GALAXY = True setelah ditinjau.')
else:
    state = load_galaxy_state()
    history_id = state.get('history_id')
    bracken_params = schema_parameters(GALAXY_SCHEMAS['Bracken'])
    bracken_results = []
    for run, run_state in state.get('runs', {}).items():
        kraken_outputs = run_state.get('outputs', {})
        report_id = find_output(kraken_outputs, ['report'])
        if not report_id:
            print(f'{run}: report Kraken2 belum tersedia; Bracken dilewati.')
            continue
        inputs = {}
        for param in bracken_params:
            label = str(param.get('label', '')).lower()
            path = param['path']
            ptype = param.get('type')
            if ptype == 'data' and 'report' in label:
                inputs[path] = {'src': 'hda', 'id': report_id}
            elif ptype == 'select' and ('kmer' in label or 'distribution' in label or 'database' in label):
                value = option_value(param, BRACKEN_DISTRIBUTION)
                if value is None:
                    raise RuntimeError(f'Distribusi {BRACKEN_DISTRIBUTION} tidak tersedia pada Bracken.')
                inputs[path] = value
            elif ptype == 'select' and ('taxonomic' in label or 'level' in label):
                inputs[path] = option_value(param, BRACKEN_LEVEL) or 'S'
            elif ptype == 'integer' and ('threshold' in label or 'minimum' in label):
                inputs[path] = BRACKEN_THRESHOLD
            elif ('kraken-style' in label or 'kraken style' in label) and ptype in {'boolean', 'select'}:
                inputs[path] = True if ptype == 'boolean' else (option_value(param, 'yes') or 'true')
        print(f'{run} input Bracken:', inputs)
        outputs = submit_once(run_state, 'bracken', BRACKEN_TOOL, inputs, history_id)
        downloaded = {}
        for name, dataset_id in outputs.items():
            wait_for_dataset(dataset_id, f'{run} Bracken {name}')
            destination = DIR_GALAXY / f'{run}.bracken.{name}.tsv'
            galaxy.datasets.download_dataset(dataset_id, file_path=str(destination), use_default_filename=False)
            downloaded[name] = str(destination)
        bracken_results.append({'run': run, **downloaded})
    manifest_bracken = pd.DataFrame(bracken_results)
    manifest_bracken.to_csv(DIR_METADATA / 'galaxy_eu_bracken_manifest.csv', index=False)
    display(manifest_bracken)


In [ ]:
# 22. Mengonversi report Kraken2 untuk Krona dan membuat visualisasi Galaxy
if not RUN_DOWNSTREAM_GALAXY:
    print('MODE REVIEW: Konversi Krona belum dijalankan.')
else:
    state = load_galaxy_state()
    history_id = state.get('history_id')
    krona_rows = []
    if KREPORT2KRONA_TOOL is None:
        raise RuntimeError('Tool konversi Kraken report ke Krona tidak ditemukan.')
    convert_schema = GALAXY_SCHEMAS['Kraken report -> Krona']
    convert_params = schema_parameters(convert_schema)
    for run, run_state in state.get('runs', {}).items():
        report_id = find_output(run_state.get('outputs', {}), ['report'])
        if not report_id:
            continue
        inputs = {}
        for param in convert_params:
            if param.get('type') == 'data':
                inputs[param['path']] = {'src': 'hda', 'id': report_id}
                break
        converted = submit_once(run_state, 'kreport2krona', KREPORT2KRONA_TOOL, inputs, history_id)
        krona_input_id = next(iter(converted.values()), None)
        krona_output_id = None
        if KRONA_TOOL and krona_input_id:
            krona_params = schema_parameters(GALAXY_SCHEMAS['Krona'])
            krona_inputs = {}
            for param in krona_params:
                if param.get('type') in {'data', 'data_collection'}:
                    krona_inputs[param['path']] = {'src': 'hda', 'id': krona_input_id}
                    break
            krona_outputs = submit_once(run_state, 'krona', KRONA_TOOL, krona_inputs, history_id)
            krona_output_id = next(iter(krona_outputs.values()), None)
        krona_rows.append({'run': run, 'krona_text_dataset': krona_input_id, 'krona_html_dataset': krona_output_id})
    manifest_krona = pd.DataFrame(krona_rows)
    manifest_krona.to_csv(DIR_METADATA / 'galaxy_eu_krona_manifest.csv', index=False)
    display(manifest_krona)


---

> **Checkpoint akhir:** tabel kelimpahan harus memiliki satu kolom untuk setiap run yang berhasil diklasifikasikan.


## 14. Tabel kelimpahan, visualisasi, dan diversitas

Tahap terakhir dijalankan di Python menggunakan berkas Bracken yang telah diunduh. Kode membentuk matriks spesies-per-sampel, menghitung kelimpahan relatif, membuat diagram komposisi, menghitung Shannon, serta matriks Bray--Curtis. Jika baru satu run yang tersedia, tabel tetap dibuat tetapi perbandingan antarsampel belum dapat ditafsirkan.


In [ ]:
# 23. Membaca seluruh hasil Bracken dan membentuk matriks kelimpahan
import math

DIR_ANALYSIS = ROOT / 'analysis'
DIR_ANALYSIS.mkdir(parents=True, exist_ok=True)

bracken_files = sorted(DIR_GALAXY.glob('*.bracken.*.tsv'))
abundance_tables = []
for path in bracken_files:
    try:
        table = pd.read_csv(path, sep='\t')
    except Exception:
        continue
    normalized = {str(c).lower(): c for c in table.columns}
    name_col = normalized.get('name') or normalized.get('taxonomy_name')
    count_col = normalized.get('new_est_reads') or normalized.get('fraction_total_reads')
    if not name_col or not count_col:
        continue
    run = path.name.split('.bracken.', 1)[0]
    values = pd.to_numeric(table[count_col], errors='coerce').fillna(0)
    abundance_tables.append(pd.DataFrame({'taxon': table[name_col].astype(str), run: values}))

if not abundance_tables:
    print('Hasil Bracken belum tersedia. Jalankan tahap Bracken terlebih dahulu.')
    abundance_matrix = pd.DataFrame()
else:
    abundance_matrix = abundance_tables[0]
    for table in abundance_tables[1:]:
        abundance_matrix = abundance_matrix.merge(table, on='taxon', how='outer')
    abundance_matrix = abundance_matrix.fillna(0).groupby('taxon', as_index=True).sum()
    abundance_matrix.to_csv(DIR_ANALYSIS / 'species_abundance_matrix.tsv', sep='\t')
    relative_abundance = abundance_matrix.div(abundance_matrix.sum(axis=0), axis=1).fillna(0)
    relative_abundance.to_csv(DIR_ANALYSIS / 'species_relative_abundance.tsv', sep='\t')
    display(abundance_matrix.head(20))


In [ ]:
# 24. Visualisasi komposisi, Shannon, dan Bray-Curtis
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib', 'seaborn'])
    import matplotlib.pyplot as plt
    import seaborn as sns

if abundance_matrix.empty:
    print('Analisis diversitas menunggu matriks kelimpahan.')
else:
    # Komposisi 15 takson paling melimpah; sisanya digabung sebagai Other.
    top_taxa = relative_abundance.sum(axis=1).nlargest(15).index
    composition = relative_abundance.loc[top_taxa].copy()
    other = 1 - composition.sum(axis=0)
    composition.loc['Other'] = other.clip(lower=0)
    ax = composition.T.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='tab20')
    ax.set_ylabel('Kelimpahan relatif')
    ax.set_xlabel('Sampel')
    ax.legend(title='Takson', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig(DIR_ANALYSIS / 'taxonomic_composition.png', dpi=200, bbox_inches='tight')
    plt.show()

    # Shannon: -sum(p * ln(p)); nilai nol tidak dimasukkan ke logaritma.
    shannon = relative_abundance.apply(
        lambda col: -sum(p * math.log(p) for p in col if p > 0), axis=0
    ).rename('shannon').to_frame()
    shannon.index.name = 'run'
    shannon.to_csv(DIR_ANALYSIS / 'shannon_diversity.tsv', sep='\t')
    display(shannon)

    # Bray-Curtis = sum(|xi-xj|) / sum(xi+xj).
    samples = abundance_matrix.columns.tolist()
    bray = pd.DataFrame(0.0, index=samples, columns=samples)
    for a in samples:
        for b in samples:
            numerator = (abundance_matrix[a] - abundance_matrix[b]).abs().sum()
            denominator = (abundance_matrix[a] + abundance_matrix[b]).sum()
            bray.loc[a, b] = numerator / denominator if denominator else 0.0
    bray.to_csv(DIR_ANALYSIS / 'bray_curtis_matrix.tsv', sep='\t')
    display(bray)
    if len(samples) > 1:
        sns.heatmap(bray, cmap='viridis', vmin=0, vmax=1, annot=len(samples) <= 12)
        plt.title('Bray-Curtis dissimilarity')
        plt.tight_layout()
        plt.savefig(DIR_ANALYSIS / 'bray_curtis_heatmap.png', dpi=200, bbox_inches='tight')
        plt.show()
    else:
        print('Bray-Curtis memerlukan minimal dua sampel untuk perbandingan.')

    print(f'Seluruh hasil analisis tersimpan di: {DIR_ANALYSIS.resolve()}')
